# HOW TO GLIMT

In [2]:
import load_secrets, os
load_secrets.load_secrets()

## JSX Requests

The JSX requests using curl have this format:

In [10]:
import os, json, requests

def jsx_request(jsxRequest):
    url = "https://glimt.nu/glimt-jsx/jsx.json?lang=en"
    
    headers = {
        "Content-Type": "application/json;charset=utf-8"
    }
    
    cookies = {
        "lumAuth": os.getenv("GLIMT_API_KEY")
    }
       
    response = requests.post(url, headers=headers, cookies=cookies, data=jsxRequest)
    
    print(response.status_code)

    return json.loads(response.text)

**Response text:** the text returned by a JSX request is itself always wrapped inside a JSON array. Therefore, below, when we say that a request returns value X, it really means that it returns [ X ] . 

If the response  text does not start by '[', i.e. is not a JSON Array, it indicates an error which is described more or less opaquely in the reply.

## Query active IFPs

To request the list of active IFPs, replace jsxRequest by:

In [17]:
def list_active_ifps():
    L = jsx_request("""[["ifps", "queryIFPs", {query: {state: "active"}, fmt: {}}]]""")
    return L[0]

In [ ]:
[["ifps", "queryIFPs", {query: {state: "active"}, fmt: {}}]]

It will return a JSON array of all active iFPs and their detailed properties, including:
* **symbol**
* **title**
* **details**: Information beyond the title of the IFP, such as what sources may be used to resolve the IFP, and/or some background information that might be useful to forecasters, &c.
* **bins**: An array of the proposed resolution outcomes

In [18]:
ifps = list_active_ifps()

200


In [29]:
len(ifps)

62

In [28]:
ifps[0]

{'id': 512,
 'type': 'bins',
 'symbol': 'War_competition_EAST_June_25',
 'state': 'closed',
 'dates': {'startDay': 20241,
  'endDay': 20270,
  'scoringStartDay': 20241,
  'scoringEndDay': 20270,
  'closedDay': 20274},
 'props': {'title': 'Will Russian troops appear in the part of Ukraine that lies to the west of the Dnieper River before the end of June?',
  'ai_title': '',
  'shortTitle': 'Will Russian troops appear in western Ukraine in June?',
  'details': '<b>Resolution:</b> The question will resolve as “yes” if Russian troops appear anywhere in the part of Ukraine to the west of the Dnieper River and this is reported in credible Ukrainian news outlets or by official Ukrainian reports. <p>\n<b>Background:</b> Ukraine is divided from north to south by the Dnieper River. Some regions, like Kyiv, Cherkasy and Kherson are also divided by the river. Most of the fighting during the war in Ukraine has taken place in the parts of Ukraine to the east of the Dnieper. As of late May 2025, ther

In [26]:
import pandas as pd
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)
df = pd.DataFrame([(ifp['id'], ifp['symbol'], ifp['props']['shortTitle']) for ifp in ifps], columns = ['id', 'challenge', 'title'])

In [27]:
df.sort_values(by=['challenge', 'title'])

,id,challenge,title
32,482,Eco_Kuzbass_Feb_25,How many more Kuzbass mines will shut down before May?
21,471,Eco_NORDSTREAM_Jan_25,Will Nord Stream be back in 2025?
25,475,Eco_RUSSIA_FUND_Jan_25,Russia to run out of cash by October?
55,505,Eco_competition_CRUDEOIL_June_25,Will the price of Brent crude oil fall below $60 in June?
53,503,Eco_competition_SANCTIONS_June_25,Will Trump impose new sanctions against Russia before end of June?
18,468,Economics_RESERVES_Jan_25,When will Russia get back its frozen foreign exchange reserves?
31,481,Economy_Shadow_Feb_25,Russian oil spill affecting EU or Nato before April 15?
8,458,Georgiainvasion,Will Russia invade Georgia in 2025?
24,474,GermanVoteShare25,Distribution of the popular vote in German elections?
5,453,Germanycoalition,Who will govern Germany?


## Submit forecast to IFP

To submit a forecast, replace jsxRequest by:

In [6]:
def submit_forecast(symbol, rationale, binProbas):
    return jsx_request(f"""[["ifps","submitAIFcst",{"ifpRef": "{symbol}","data": {"probas": {binProbas} },"reasoning": {rationale}}]]""")

where you should replace 
* **symbol** with the symbol of the IFP for which you are submitting a forecast bin
* **binProbas** with a JSON array of probabilities adding to 1.0, and such that each one corresponds to the IFP's bin (i.e. outcome) at the same index. For example, an IFP with 4 outcomes could accept [0.2, 0.6, 0, 0.2] where 0.6 is the probability you assign to the second outcome.
* **rationale** with your reasoning

It will return a JSON object containing the full description of your submitted forecast, including the forecast ID.